In [0]:
!pip install yfinance certifi lxml

In [0]:
import yfinance as yf
import pandas as pd
import requests
import datetime
import time

## II. Retrieve the List of Stock Symbols

In [0]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
tables = pd.read_html(url)
sp500_table = tables[0]
sp500_symbols = sp500_table['Symbol'].tolist()
#display(sp500_table['Symbol'])

## III. Retrieve the Historical News Bronze Data and Store into the DBFS

In [0]:
API_KEY_FH = "cukgc11r01qo08i88f0gcukgc11r01qo08i88f10"
end_date = pd.Timestamp.now().strftime("%Y-%m-%d")
#start_date = (pd.Timestamp.now() - pd.DateOffset(years=1)
              #).strftime("%Y-%m-%d")
start_date = (pd.Timestamp.now() - pd.Timedelta(days=1)).strftime("%Y-%m-%d")

def fetch_historical_news_dataframe_FH(symbols):
  # Dictionary to store historical stock news data
  stock_news_data = []
  for symbol in symbols:
    try:
      url_news = f"https://finnhub.io/api/v1/company-news?symbol={symbol}&from={start_date}&to={end_date}&token={API_KEY_FH}"
      response = requests.get(url_news)
      if response.status_code == 200:
        data = response.json()
        if data:
          data_news_df = pd.DataFrame(data)
          data_news_df['datetime'] = pd.to_datetime(data_news_df['datetime'], unit='s')
          filtered_df = data_news_df[data_news_df['datetime'].dt.strftime("%Y-%m-%d") == end_date]
          filtered_df['symbol'] = str(symbol)
          stock_news_data.append(filtered_df)
          time.sleep(1)
        else:
          print(f"⚠️ No news data found for {symbol}")
      else:
          print(
              f"⚠️ Failed to fetch data for {symbol}, Status Code: {response.status_code}")
    except Exception as e:
      print(f"❌ Error fetching {symbol}: {e}")

    print(f"✅ Processed {symbol}")

  stock_news_data = pd.concat(stock_news_data)
  stock_news_data.reset_index(drop=True, inplace=True)
  return stock_news_data

stock_news_data = fetch_historical_news_dataframe_FH(sp500_symbols)

In [0]:
display(stock_news_data)

In [0]:
jdbc_url = "jdbc:sqlserver://capstone-database-server.database.windows.net:1433;database=writedatabasesilverlayer;"
connection_properties = {
    "user": "capstonedioxieteam",
    "password": "Connhenbeo1@",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

In [0]:
old_data = spark.read.jdbc(
    url=jdbc_url,
    table="Bronze.Historical_Stock_News",
    properties=connection_properties
)
start_id = old_data.count() + 2
stock_news_data['id'] = range(start_id, start_id + len(stock_news_data))
stock_news_data.to_csv('/dbfs/FileStore/Bronze/Historical_Stock_News_Bronze.csv', index=False)


In [0]:
desired_order = [
    'id',
    'category',
    'datetime',
    'headline',
    'image',
    'related',
    'source',
    'summary',
    'url',
    'symbol'
]

stock_news_data_df = spark.read.option("header", True) \
    .option("inferSchema", True) \
    .option("multiLine", True) \
    .option("escape", "\"") \
    .csv("/FileStore/Bronze/Historical_Stock_News_Bronze.csv")

stock_news_data_df = stock_news_data_df.select(desired_order)
stock_news_data_df.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "Bronze.Historical_Stock_News") \
    .option("user", connection_properties["user"]) \
    .option("password", connection_properties["password"]) \
    .option("driver", connection_properties["driver"]) \
    .mode("append") \
    .option("batchsize", 10000) \
    .option("numPartitions", 8) \
    .save()

print("Data successfully written to Azure SQL Database.")